In [ ]:
# =========================
# Faster-Whisper ASR batch transcription
# =========================

!pip -q install faster-whisper pandas

import os
import json
import time
import zipfile
import shutil
import pandas as pd
from pathlib import Path
from google.colab import files
from faster_whisper import WhisperModel


MODEL_SIZE = "base"
LANGUAGE = "ru"
USE_VAD = True

DEVICE = "cuda"
COMPUTE_TYPE = "float16"
CPU_THREADS = 4


AUDIO_EXTENSIONS = {
    ".mp3", ".wav", ".m4a", ".ogg", ".opus", ".flac", ".aac", ".webm", ".mp4"
}

WORK_DIR = Path("/content/asr_batch")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "results"

RESULTS_CSV = OUTPUT_DIR / "transcripts.csv"
RESULTS_JSON = OUTPUT_DIR / "all_transcripts.json"
RESULTS_TXT = OUTPUT_DIR / "all_transcripts.txt"
RESULTS_ZIP = Path("/content/asr_results.zip")


if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("Загрузи zip-архив с папкой аудиофайлов...")
uploaded = files.upload()

zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]

if not zip_files:
    raise ValueError("Нужно загрузить именно .zip архив с аудиофайлами.")

zip_path = Path(zip_files[0])

print(f"Распаковываю: {zip_path}")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(INPUT_DIR)


audio_files = sorted([
    p for p in INPUT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
])

if not audio_files:
    raise ValueError("В архиве не найдено аудиофайлов поддерживаемых форматов.")

print(f"Найдено аудиофайлов: {len(audio_files)}")



print("\nЗагружаю модель Faster-Whisper...")
model_load_start = time.perf_counter()

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    cpu_threads=CPU_THREADS
)

model_load_seconds = time.perf_counter() - model_load_start


rows = []
json_items = []

print("\nНачинаю транскрибацию...")
transcription_start = time.perf_counter()

for i, audio_path in enumerate(audio_files, start=1):
    print(f"[{i}/{len(audio_files)}] {audio_path.name}")

    segments, info = model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        beam_size=5,
        vad_filter=USE_VAD
    )

    full_text_parts = []

    for seg in segments:
        text = seg.text.strip()
        if text:
            full_text_parts.append(text)

    full_text = " ".join(full_text_parts).strip()
    relative_path = audio_path.relative_to(INPUT_DIR)

    duration_seconds = round(getattr(info, "duration", 0), 2)

    item = {
        "file": str(relative_path),
        "duration_seconds": duration_seconds,
        "detected_language": getattr(info, "language", None),
        "language_probability": getattr(info, "language_probability", None),
        "text": full_text
    }

    json_items.append(item)
    rows.append(item)

transcription_seconds = time.perf_counter() - transcription_start



total_audio_seconds = sum(item["duration_seconds"] for item in json_items)

result_json = {
    "settings": {
        "model_size": MODEL_SIZE,
        "language": LANGUAGE,
        "vad_filter": USE_VAD,
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "cpu_threads": CPU_THREADS
    },
    "summary": {
        "files_count": len(audio_files),
        "total_audio_seconds": round(total_audio_seconds, 2),
        "model_load_seconds": round(model_load_seconds, 2),
        "total_transcription_seconds": round(transcription_seconds, 2),
        "real_time_factor": round(transcription_seconds / total_audio_seconds, 3) if total_audio_seconds > 0 else None
    },
    "transcripts": json_items
}


with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(result_json, f, ensure_ascii=False, indent=2)

with open(RESULTS_TXT, "w", encoding="utf-8") as f:
    f.write("ИТОГИ ТРАНСКРИБАЦИИ\n")
    f.write("====================\n")
    f.write(f"Количество файлов: {len(audio_files)}\n")
    f.write(f"Суммарная длительность аудио: {round(total_audio_seconds, 2)} сек.\n")
    f.write(f"Время загрузки модели: {round(model_load_seconds, 2)} сек.\n")
    f.write(f"Общее время транскрибации: {round(transcription_seconds, 2)} сек.\n")
    if total_audio_seconds > 0:
        f.write(f"Real-time factor: {round(transcription_seconds / total_audio_seconds, 3)}\n")
    f.write("\n\n")

    for item in json_items:
        f.write(f"ФАЙЛ: {item['file']}\n")
        f.write("-" * 80 + "\n")
        f.write(item["text"] + "\n\n")



df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")



if RESULTS_ZIP.exists():
    RESULTS_ZIP.unlink()

shutil.make_archive(
    base_name=str(RESULTS_ZIP.with_suffix("")),
    format="zip",
    root_dir=OUTPUT_DIR
)



print("\nГотово!")
print(f"Количество файлов: {len(audio_files)}")
print(f"Суммарная длительность аудио: {round(total_audio_seconds, 2)} сек.")
print(f"Время загрузки модели: {round(model_load_seconds, 2)} сек.")
print(f"Общее время транскрибации всех файлов: {round(transcription_seconds, 2)} сек.")

if total_audio_seconds > 0:
    print(f"Real-time factor: {round(transcription_seconds / total_audio_seconds, 3)}")
    print("RTF < 1 означает быстрее реального времени, RTF > 1 — медленнее.")

print(f"\nJSON: {RESULTS_JSON}")
print(f"TXT: {RESULTS_TXT}")
print(f"CSV: {RESULTS_CSV}")
print(f"Архив: {RESULTS_ZIP}")

display(df[["file", "duration_seconds", "text"]])

files.download(str(RESULTS_ZIP))

Загрузи zip-архив с папкой аудиофайлов...


Saving inassist_synthetic_asr_dataset (1).zip to inassist_synthetic_asr_dataset (1).zip
Распаковываю: inassist_synthetic_asr_dataset (1).zip
Найдено аудиофайлов: 100

Загружаю модель Faster-Whisper...

Начинаю транскрибацию...
[1/100] 001.mp3
[2/100] 002.mp3
[3/100] 003.mp3
[4/100] 004.mp3
[5/100] 005.mp3
[6/100] 006.mp3
[7/100] 007.mp3
[8/100] 008.mp3
[9/100] 009.mp3
[10/100] 010.mp3
[11/100] 011.mp3
[12/100] 012.mp3
[13/100] 013.mp3
[14/100] 014.mp3
[15/100] 015.mp3
[16/100] 016.mp3
[17/100] 017.mp3
[18/100] 018.mp3
[19/100] 019.mp3
[20/100] 020.mp3
[21/100] 021.mp3
[22/100] 022.mp3
[23/100] 023.mp3
[24/100] 024.mp3
[25/100] 025.mp3
[26/100] 026.mp3
[27/100] 027.mp3
[28/100] 028.mp3
[29/100] 029.mp3
[30/100] 030.mp3
[31/100] 031.mp3
[32/100] 032.mp3
[33/100] 033.mp3
[34/100] 034.mp3
[35/100] 035.mp3
[36/100] 036.mp3
[37/100] 037.mp3
[38/100] 038.mp3
[39/100] 039.mp3
[40/100] 040.mp3
[41/100] 041.mp3
[42/100] 042.mp3
[43/100] 043.mp3
[44/100] 044.mp3
[45/100] 045.mp3
[46/100] 046.mp3


,file,duration_seconds,text
0,audio/001.mp3,4.92,Постав встречу с куратором завтра в 14 часов н...
1,audio/002.mp3,4.15,Добавь звонок с командой в пятницу в 11 часов.
2,audio/003.mp3,4.20,Запланирую тренировку сегодня вечером на полто...
3,audio/004.mp3,4.70,Создай события разбор курсовой на понедельник ...
4,audio/005.mp3,4.68,Постав на поминание про плату интернета на 25 ...
...,...,...,...
95,audio/096.mp3,3.46,Когда у меня следующая консультация?
96,audio/097.mp3,3.00,Покажи все события на выходных.
97,audio/098.mp3,3.58,Есть ли пересечения в расписании завтра?
98,audio/099.mp3,3.31,Покажи расписание на следующую среду.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>